## Test Build NetCDF
Test building a netCDF similar to that used in LM4 from reanalysis data. As will the production script, I will use NLDAS data, but just download a month of air temperature data. I need to ensure that I use the same variable names as is in the target NetCDF.

In [10]:
import requests
import pandas as pd
import xarray as xr
from datetime import datetime
from io import StringIO

Load swamp angel data for comparison with the dataset I am building

In [3]:
# Load the data
data = xr.open_dataset('../data/swa/swa_insitu_forcing.nc')

In [4]:
data

<xarray.Dataset> Size: 4MB
Dimensions:           (time: 87648, latitude: 1, longitude: 1)
Coordinates:
  * time              (time) datetime64[ns] 701kB 2005-10-01T07:59:56 ... 201...
  * latitude          (latitude) float64 8B 37.91
  * longitude         (longitude) float64 8B -107.7
Data variables:
    LWdown            (time, latitude, longitude) float32 351kB ...
    Psurf             (time, latitude, longitude) float32 351kB ...
    Qair              (time, latitude, longitude) float32 351kB ...
    Rainf             (time, latitude, longitude) float32 351kB ...
    Snowf             (time, latitude, longitude) float32 351kB ...
    SWdown            (time, latitude, longitude) float32 351kB ...
    Tair              (time, latitude, longitude) float32 351kB ...
    Wind              (time, latitude, longitude) float32 351kB ...
    reference_height  (latitude, longitude) float64 8B ...
    canopy_height     (latitude, longitude) float64 8B ...

For this testing script, I will use the Tair variable. I will create some fake site data that is analogous to what is found in the PNNL snotel data. Note that the starting and ending dates are in an "American" format.

In [8]:
fake_snotel = {'latitude': 41.89, 'longitude': -111.57, 'elevation': 3000, 'start_date': '10/1/2010', 'end_date': '11/1/2010'}
forcing_name = 'Tair'

#### Get NLDAS for testing

Pretty much all variable names in the target netCDF match the NLDAS variable names except for wind.

In [9]:
# Convert start and end dates to datetime objects
nldas_start = datetime.strptime(fake_snotel['start_date'], '%m/%d/%Y')
nldas_end = datetime.strptime(fake_snotel['end_date'], '%m/%d/%Y')

In [13]:
# Define the base URL for the NLDAS Data Rods API
base_url = 'https://hydro1.gesdisc.eosdis.nasa.gov/daac-bin/access/timeseries.cgi'

# Define the parameters for the API request
params = {
    'variable': f'NLDAS2:NLDAS_FORA0125_H_v2.0:{forcing_name}',  # Example variable: 2-meter temperature
    'startDate': datetime.strftime(nldas_start,'%Y-%m-%dT00'),  # Start date in YYYY-MM-DDTHH format
    'endDate': datetime.strftime(nldas_end,'%Y-%m-%dT00'),    # End date in YYYY-MM-DDTHH format
    'location': f'GEOM:POINT({fake_snotel['longitude']}, {fake_snotel['latitude']})',  # Location as a point (longitude, latitude), note that the space in lat is important!!
    'type': 'asc2'  # Response format (e.g., asc2 for ASCII)
}

# Make the GET request to the API
response = requests.get(base_url, params=params)

# Check if the request was successful
if response.status_code == 200:
    # Print the response content
    nldas_csv = StringIO(response.text)
else:
    print(f"Error: {response.status_code}")

In [14]:
# Review the first few lines of the data to determine how many rows to skip for the header
sample_lines = nldas_csv.getvalue().split('\n')[:15]
for line in sample_lines:
    print(line)

Metadata for Requested Time Series:

prod_name=NLDAS_FORA0125_H_v2.0
param_short_name=Tair
param_name=2-meter above ground Temperature
unit=K
begin_time=2010-10-01T00
end_time=2010-11-01T00
lat= 41.9375
lon=-111.5625
Request_time=2025-03-06 19:22:01 GMT

Date&Time               Data
2010-10-01T00:00:00	294.29
2010-10-01T01:00:00	291.8


In [15]:
# Based on the inspection, I should skip the first 12 rows of data
skip_rows = 12

# Load the data into a Pandas DataFrame, skipping the necessary rows
nldas_csv.seek(0)  # Reset the StringIO object to the beginning
nldas_data = pd.read_csv(nldas_csv, delimiter=r'\s+', skiprows=skip_rows, index_col=0, parse_dates=True)

In [16]:
# Rename column to match forcing name
nldas_data.columns = [forcing_name]

For the sake of testing, I will create a second column in the dataframe, just to have another variable when making the NetCDF

In [17]:
nldas_data['Qair'] = nldas_data['Tair']

In [28]:
# Rename index to 'time'
nldas_data.index.name = 'time'

### Build the NetCDF

In [22]:
# Add on latitude and longitude columns to the dataframe so they will become coordinates in the xarray dataset
nldas_data['latitude'] = fake_snotel['latitude']
nldas_data['longitude'] = fake_snotel['longitude']

In [29]:
forcing_dataset = xr.Dataset.from_dataframe(nldas_data)
forcing_dataset = forcing_dataset.set_coords(['latitude', 'longitude'])

In [30]:
forcing_dataset

<xarray.Dataset> Size: 30kB
Dimensions:    (time: 745)
Coordinates:
    latitude   (time) float64 6kB 41.89 41.89 41.89 41.89 ... 41.89 41.89 41.89
    longitude  (time) float64 6kB -111.6 -111.6 -111.6 ... -111.6 -111.6 -111.6
  * time       (time) datetime64[ns] 6kB 2010-10-01 ... 2010-11-01
Data variables:
    Tair       (time) float64 6kB 294.3 291.8 289.3 286.8 ... 278.0 277.3 276.7
    Qair       (time) float64 6kB 294.3 291.8 289.3 286.8 ... 278.0 277.3 276.7